# Fine-tuning PhoBERT with LoRA

PhoBERT-base-v2 from Hugging Face, fine-tuned on the reviews by masked language modelling: the reviews have no labels, so it learns their language by guessing words hidden from it. With LoRA, PhoBERT's own weights stay as they are and only small added matrices in its attention are trained.

In [1]:
import pandas as pd

data = pd.read_csv("data/bert_food_history.csv", usecols=["content"])
# Only written reviews: purchases not rated, or rated with just the stars, are dropped
data = data[data["content"].fillna("").str.strip() != ""].reset_index(drop=True)

print(f"{len(data)} reviews")
data.head()

6744 reviews


,content
0,"Ăn được đó, ít quá, không đã, giá hợp lý, soco..."
1,"Ngon hết nấc, fan cà phê như anh thì khỏi chê,..."
2,"Yêu ghê 🥰 Đáng đồng tiền, vị đậm đà, em vốn th..."
3,"Ok thôi, đắt so với chất lượng, anh mê hải sản..."
4,"Good, nêm hơi mặn. Cảm ơn em nha"


In [2]:
data.shape

(6744, 1)

In [3]:
from sklearn.model_selection import train_test_split

# 90% to train on, 10% held back to check it on reviews it hasn't seen
train, val = train_test_split(data, test_size=0.1, random_state=0)
print("training:", train.shape, "validation:", val.shape)

training: (6069, 1) validation: (675, 1)


In [4]:
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

PHOBERT = "vinai/phobert-base-v2"
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

# PhoBERT reads word-segmented text ("Trà_sữa trân_châu"): segment with underthesea's word_tokenize first.
# Inputs up to 256 tokens.
tokenizer = AutoTokenizer.from_pretrained(PHOBERT)
# On MPS (Apple GPUs) the default attention can't do dropout, which training needs: use the plain one there
attention = "eager" if device == "mps" else "sdpa"
# With its masked-word head: what it's fine-tuned on
model = AutoModelForMaskedLM.from_pretrained(PHOBERT, attn_implementation=attention).to(device)
model.train()  # dropout on, every weight trainable

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{PHOBERT} on {device}: {trainable / 1e6:.0f}M trainable parameters")

W1128 13:41:38.411000 12509 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

vinai/phobert-base-v2 on mps: 135M trainable parameters


In [5]:
from peft import LoraConfig, PeftModel, get_peft_model

# LoRA: PhoBERT's weights frozen, and next to each attention query and value layer a pair of small
# matrices (rank 8) that learn the change instead: well under 1% of the parameters to train
LORA = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1, target_modules=["query", "value"])
if not isinstance(model, PeftModel):  # run twice, it would add LoRA on top of LoRA
    model = get_peft_model(model, LORA)
model.train()
model.print_trainable_parameters()

trainable params: 294,912 || all params: 135,358,721 || trainable%: 0.2179


In [6]:
from underthesea import word_tokenize

texts = ["Trà sữa trân châu đường đen", "Matcha latte ít ngọt, ngon xỉu"]
batch = tokenizer([word_tokenize(t, format="text") for t in texts], padding=True, truncation=True, max_length=256, return_tensors="pt").to(device)
print({name: tuple(tensor.shape) for name, tensor in batch.items()})  # input: (batch, seq_len)
with torch.no_grad():
    print("last_hidden_state:", tuple(model.roberta(**batch).last_hidden_state.shape))  # model.roberta: PhoBERT without the head  # output: (batch, seq_len, 768)

{'input_ids': (2, 11), 'attention_mask': (2, 11)}
last_hidden_state: (2, 11, 768)


In [7]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling


def encode(texts):
    """Word-segmented as PhoBERT expects and cut to its 256 tokens, not padded yet"""
    return [tokenizer(word_tokenize(t, format="text"), truncation=True, max_length=256) for t in texts]


# Pads each batch to its longest review, and hides 15% of the tokens for the model to guess (labels: the hidden words)
collator = DataCollatorForLanguageModeling(tokenizer, mlm_probability=0.15)
train_loader = DataLoader(encode(train["content"]), batch_size=16, shuffle=True, collate_fn=collator)

# The validation reviews are masked once, so every check scores the same hidden words
torch.manual_seed(0)
val_batches = list(DataLoader(encode(val["content"]), batch_size=32, collate_fn=collator))

batch = next(iter(train_loader))
print(f"{len(train_loader)} training batches, {len(val_batches)} validation batches")
print({name: tuple(tensor.shape) for name, tensor in batch.items()})  # each (batch, seq_len)

380 training batches, 22 validation batches
{'input_ids': (16, 28), 'attention_mask': (16, 28), 'labels': (16, 28)}


In [9]:
import math

from tqdm.auto import tqdm
from transformers import get_linear_schedule_with_warmup

EPOCHS = 5
# Only the LoRA matrices train, and at a higher learning rate than full fine-tuning (2e-5) would use
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-4, weight_decay=0.01)
steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=steps // 10, num_training_steps=steps)


@torch.no_grad()
def validation_loss():
    model.eval()
    losses = [model(**{k: v.to(device) for k, v in b.items()}).loss.item() for b in val_batches]
    model.train()
    return sum(losses) / len(losses)


print(f"before: validation loss {validation_loss():.3f}")
for epoch in range(1, EPOCHS + 1):
    total = 0.0
    for batch in tqdm(train_loader, desc=f"epoch {epoch}"):
        loss = model(**{k: v.to(device) for k, v in batch.items()}).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total += loss.item()
    val_loss = validation_loss()
    print(f"epoch {epoch}: training loss {total / len(train_loader):.3f}, validation loss {val_loss:.3f} (perplexity {math.exp(val_loss):.1f})")

before: validation loss 1.184


epoch 1:   0%|          | 0/380 [00:00<?, ?it/s]

epoch 1: training loss 1.433, validation loss 1.070 (perplexity 2.9)


epoch 2:   0%|          | 0/380 [00:00<?, ?it/s]

epoch 2: training loss 1.264, validation loss 0.922 (perplexity 2.5)


epoch 3:   0%|          | 0/380 [00:00<?, ?it/s]

epoch 3: training loss 1.112, validation loss 0.800 (perplexity 2.2)


epoch 4:   0%|          | 0/380 [00:00<?, ?it/s]

epoch 4: training loss 0.976, validation loss 0.741 (perplexity 2.1)


epoch 5:   0%|          | 0/380 [00:00<?, ?it/s]

epoch 5: training loss 0.930, validation loss 0.724 (perplexity 2.1)


In [ ]:
SAVE_TO = "models/phobert-lora"
model.save_pretrained(SAVE_TO)
tokenizer.save_pretrained(SAVE_TO)

('models/phobert-lora/tokenizer_config.json',
 'models/phobert-lora/vocab.txt',
 'models/phobert-lora/bpe.codes',
 'models/phobert-lora/added_tokens.json')